# KV Cache Precision Experiments

This notebook walks through the full Topic 2 investigation:
1. Load a model and inspect the raw KV cache
2. Measure memory footprint at different precisions
3. Quantify the error introduced by compression
4. Sweep context lengths to see how FP8/INT8 savings scale
5. Simulate concurrency to find the OOM ceiling
6. Measure latency (TPOT) across configurations

**Note:** We use GPT-2 here so everything runs on a laptop/Colab CPU.  
On the server, swap `model_id` to `meta-llama/Meta-Llama-3-8B-Instruct` and `device` to `cuda`.

In [ ]:
import sys
sys.path.insert(0, '../src')

from llm_serving.kv_cache import (
    load_model,
    extract_kv_cache,
    profile_kv_cache,
    print_cache_profile,
    convert_kv_cache_precision,
    compare_precisions,
    measure_quantization_error,
    measure_generation_latency,
    simulate_batch_concurrency,
    sweep_context_lengths,
)
from llm_serving.plotting import (
    plot_cache_size_comparison,
    plot_concurrency_throughput,
    plot_quantization_error,
    plot_precision_memory_bar,
    plot_latency_comparison,
)
import torch
print('All imports successful!')

## 1. Load the Model

Using GPT-2 (124M params) for local development.  
Change `model_id` and `device` when you move to the server.

In [ ]:
# ------ CONFIGURATION (change these on the server) ------
MODEL_ID = "gpt2"                    # server: "meta-llama/Meta-Llama-3-8B-Instruct"
DEVICE = "cpu"                       # server: "cuda"
# --------------------------------------------------------

model, tokenizer = load_model(MODEL_ID, DEVICE)

## 2. Inspect the Raw KV Cache

Run a single forward pass and look at what the KV cache actually contains —  
its shape, dtype, and memory footprint.

In [ ]:
prompt = "The future of artificial intelligence is"

kv_cache, outputs = extract_kv_cache(model, tokenizer, prompt, DEVICE)

profile = profile_kv_cache(kv_cache)
print("\n--- KV Cache Profile ---")
print_cache_profile(profile)

## 3. Compare Precision Formats

Simulate compressing the cache to FP16 and INT8 (standing in for FP8).  
Measure exactly how many bytes we save and how many more tokens we could  
theoretically fit in a 10 GB memory budget.

In [ ]:
# Compare against FP16
comp_fp16 = compare_precisions(kv_cache, torch.float16)
print(f"FP32 -> FP16:")
print(f"  Original: {comp_fp16.original_size_mb:.4f} MB")
print(f"  Compressed: {comp_fp16.compressed_size_mb:.4f} MB")
print(f"  Saved: {comp_fp16.memory_saved_pct:.1f}%")
print(f"  Max tokens in 10GB (original): {comp_fp16.max_seq_at_original:,}")
print(f"  Max tokens in 10GB (compressed): {comp_fp16.max_seq_at_compressed:,}")

print()

# Compare against INT8 (simulating FP8)
comp_int8 = compare_precisions(kv_cache, torch.int8)
print(f"FP32 -> INT8 (simulating FP8):")
print(f"  Original: {comp_int8.original_size_mb:.4f} MB")
print(f"  Compressed: {comp_int8.compressed_size_mb:.4f} MB")
print(f"  Saved: {comp_int8.memory_saved_pct:.1f}%")
print(f"  Max tokens in 10GB (original): {comp_int8.max_seq_at_original:,}")
print(f"  Max tokens in 10GB (compressed): {comp_int8.max_seq_at_compressed:,}")

In [ ]:
# Visualise the comparison
plot_precision_memory_bar({"FP16": comp_fp16, "INT8": comp_int8})

## 4. Quantization Error Analysis

Measure how much numerical error INT8 quantization introduces per layer.  
This is our proxy for output quality — if certain layers have high error,  
that's where quality degradation will show up first.

In [ ]:
errors = measure_quantization_error(kv_cache, torch.int8)

print("Per-layer quantization error (INT8):")
for e in errors:
    print(f"  Layer {e['layer']:2d}: Key MSE={e['key_mse']:.6f}, "
          f"Value MSE={e['value_mse']:.6f}, "
          f"Key MaxErr={e['key_max_abs_error']:.4f}")

plot_quantization_error(errors, "INT8")

## 5. Context Length Sweep

Measure how KV cache size grows with context length, and how much  
INT8/FP8 compression saves at each length.  

This answers: *"How does the benefit change with context length?"*

In [ ]:
# GPT-2 max context is 1024. On the server with Llama-3, use [512, 1024, 2048, 4096, 8192]
context_lengths = [32, 64, 128, 256, 512]

print("Sweeping context lengths...")
ctx_results = sweep_context_lengths(model, tokenizer, context_lengths, DEVICE)

plot_cache_size_comparison(ctx_results)

## 6. Batch Concurrency Simulation

Simulate increasing the number of concurrent users by growing the batch size.  
This finds the OOM point — your concurrency ceiling.

On a laptop/CPU this won't OOM (it'll just get slow), but on the GPU server  
you will see it crash at a specific batch size. That crash point is your data.

In [ ]:
batch_sizes = [1, 2, 4, 8, 16]
# On the server with GPU: batch_sizes = [1, 2, 4, 8, 16, 32, 64, 128]

prompt = "Explain the concept of attention in transformers."

print("Running batch concurrency sweep...")
batch_results = simulate_batch_concurrency(
    model, tokenizer, prompt,
    batch_sizes=batch_sizes,
    max_new_tokens=20,
    device=DEVICE,
)

plot_concurrency_throughput(batch_results)

## 7. Latency Measurement (TPOT / TTFT)

Measure how fast the model generates tokens. This is the baseline  
that speculative decoding (Topic 3) will try to beat.

In [ ]:
prompt = "The key challenge in deploying large language models is"

result = measure_generation_latency(
    model, tokenizer, prompt,
    max_new_tokens=50,
    device=DEVICE,
)

print(f"--- Generation Latency ---")
print(f"  Tokens generated: {result.num_tokens_generated}")
print(f"  Total time: {result.total_time_sec:.3f}s")
print(f"  Time to first token (TTFT): {result.time_to_first_token_sec:.3f}s")
print(f"  Time per output token (TPOT): {result.time_per_output_token_ms:.1f}ms")
print(f"  Tokens/sec: {result.tokens_per_sec:.1f}")

## Next Steps

When you get server access:
1. Change `MODEL_ID` to `meta-llama/Meta-Llama-3-8B-Instruct`
2. Change `DEVICE` to `cuda`
3. Expand `context_lengths` to `[512, 1024, 2048, 4096, 8192]`
4. Expand `batch_sizes` to `[1, 2, 4, 8, 16, 32, 64, 128]`
5. Run MMLU / HumanEval benchmarks to check quality degradation
6. Use vLLM with `--kv-cache-dtype fp8` for native FP8 support